In [2]:
import os, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
import argparse
import scanpy as sc

def one_hot_encode(class_list, unique_classes):
    # Map each class in class_list to its index in unique_classes
    class_to_index = {cls: idx for idx, cls in enumerate(unique_classes)}
    indices = [class_to_index[cls] for cls in class_list]

    # Vectorized one-hot creation
    one_hot = np.eye(len(unique_classes), dtype=int)[indices]
    return one_hot

def label_encode(class_list, unique_classes):
    # Map each class to a positive integer (starting at 1)
    class_to_index = {cls: idx for idx, cls in enumerate(unique_classes)}
    return np.array([class_to_index[cls] for cls in class_list], dtype=int)

train_adata = sc.read("data/scgpt_merhh_train.h5ad")
test_adata = sc.read("data/scgpt_merhh_test.h5ad")

#def set_seed(seed=42):
#    import random
#    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
#    if torch.cuda.is_available():
#        torch.cuda.manual_seed_all(seed)
#set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# ========= 1) Label encoding & data split =========

X_train_full = np.array(train_adata.X).astype('float32')
train_label = train_adata.obs['ground_truth']

X_test = np.array(test_adata.X).astype('float32')
test_label = test_adata.obs['ground_truth']

classes_ = np.unique(np.concatenate([np.unique(train_label), np.unique(test_label)]))

y_train_full = label_encode(train_label, classes_)
y_test = label_encode(test_label, classes_)



X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.2, random_state=42
)
X_tr_t   = torch.tensor(X_tr,   dtype=torch.float32)
X_val_t  = torch.tensor(X_val,  dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_tr_t   = torch.tensor(y_tr,   dtype=torch.long)
y_val_t  = torch.tensor(y_val,  dtype=torch.long)
y_test_t = torch.tensor(y_test, dtype=torch.long)
in_dim = X_tr.shape[1]
num_classes = len(classes_)
# ========= 2) Model =========
class MLPClassifier(nn.Module):
    def __init__(self, in_dim, num_classes, h1=512, h2=256, h3=128, p_drop=0.3):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, h1)
        self.bn1 = nn.BatchNorm1d(h1)
        self.fc2 = nn.Linear(h1, h2)
        self.bn2 = nn.BatchNorm1d(h2)
        #self.fc3 = nn.Linear(h2, h3)
        #self.bn3 = nn.BatchNorm1d(h3)
        self.out = nn.Linear(h2, num_classes)
        self.p = p_drop
    def forward(self, x):
        x = F.relu(self.bn1(self.fc1(x)))
        x = F.dropout(x, p=self.p, training=self.training)
        x = F.relu(self.bn2(self.fc2(x)))
        x = F.dropout(x, p=self.p, training=self.training)
        #x = F.relu(self.bn3(self.fc3(x)))
        #x = F.dropout(x, p=self.p, training=self.training)
        return self.out(x)
# ========= 3) Helpers =========
def make_loaders(batch_size):
    train_ds = TensorDataset(X_tr_t, y_tr_t)
    val_ds   = TensorDataset(X_val_t, y_val_t)
    test_ds  = TensorDataset(X_test_t, y_test_t)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total += loss.item() * xb.size(0)
    return total / len(loader.dataset)
@torch.no_grad()
def eval_loss(model, loader, criterion):
    model.eval()
    total = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)
        total += loss.item() * xb.size(0)
    return total / len(loader.dataset)
@torch.no_grad()
def eval_metrics(model, loader):
    model.eval()
    all_preds, all_true = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        logits = model(xb)
        preds = logits.argmax(1).cpu().numpy()
        all_preds.extend(preds); all_true.extend(yb.numpy())
    acc = accuracy_score(all_true, all_preds)
    f1m = f1_score(all_true, all_preds, average='macro')
    return acc, f1m, np.array(all_true), np.array(all_preds)
# ========= 4) Optuna objective =========
def objective(trial: optuna.Trial):
    # --- hyperparams to tune ---
    h1 = trial.suggest_int("h1", 10, in_dim)
    h2 = trial.suggest_int("h2", 10, 256)
    h3 = 10#trial.suggest_int("h3", 10, 32)
    drop = trial.suggest_float('dropout', 0.1, 0.6)
    lr = trial.suggest_float('lr', 1e-5, 1e-1, log=True)
    wd = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)
    batch_size = 64#trial.suggest_categorical('batch_size', [64, 128, 256, 512])
    use_class_weights = trial.suggest_categorical('use_class_weights', [True, False])
    train_loader, val_loader, _ = make_loaders(batch_size)
    model = MLPClassifier(in_dim, num_classes, h1=h1, h2=h2, h3=h3, p_drop=drop).to(device)
    # class weights (optional)
    if use_class_weights:
        class_counts = np.bincount(y_tr, minlength=num_classes)
        cw = (class_counts.sum() / np.maximum(class_counts, 1)).astype(np.float32)
        cw = cw / cw.mean()
        weight = torch.tensor(cw, dtype=torch.float32, device=device)
    else:
        weight = None
    criterion = nn.CrossEntropyLoss(weight=weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    best_val_loss = float("inf")
    best_state = None
    patience, bad = 8, 0
    max_epochs = 50
    for epoch in range(1, max_epochs + 1):
        tr_loss = train_epoch(model, train_loader, optimizer, criterion)
        val_loss = eval_loss(model, val_loader, criterion)
        trial.report(val_loss, step=epoch)
        
        if val_loss < best_val_loss - 1e-4:
            best_val_loss, bad = val_loss, 0
            # store CPU copy to avoid GPU memory hold
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
        # Early stopping
        #if bad >= patience:
        #    break
    if best_state is None:
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    # keep the best checkpoint + params on the trial
    trial.set_user_attr('best_state_dict', best_state)
    trial.set_user_attr('best_val_loss', best_val_loss)
    if trial.should_prune():
            raise optuna.TrialPruned()
    return best_val_loss  # minimize
# ========= 5) Run study =========
study = optuna.create_study(
    direction='minimize',
    sampler=TPESampler(seed=0, n_startup_trials=10),
    pruner=MedianPruner(n_warmup_steps=5)
)
n_trials = 10  # tweak as you like
study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
print('Best trial:', study.best_trial.number)
print('Best value (val_loss):', study.best_trial.value)
print('Best params:', study.best_trial.params)
# ========= 6) Restore best model & evaluate on TEST =========
best_params = study.best_trial.params
best_state = study.best_trial.user_attrs['best_state_dict']

best_model = MLPClassifier(
    in_dim, num_classes,
    h1=best_params["h1"], h2=best_params['h2'], h3=10,
    p_drop=best_params['dropout']
).to(device)
best_model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
# Build loaders with the tuned batch size (for evaluation it doesn’t matter much)
_, val_loader, test_loader = make_loaders(64)#(best_params['batch_size'])
val_acc, val_f1, _, _ = eval_metrics(best_model, val_loader)
test_acc, test_f1, y_true, y_pred = eval_metrics(best_model, test_loader)
print("Val_Acc\tVal_F1\tTest_Acc\tTest_F1")
print(f"{val_acc:.4f}\t{val_f1:.4f}\t{test_acc:.4f}\t{test_f1:.4f}")
print(classification_report(y_true, y_pred, target_names=classes_, digits=4))
'''# ========= 7) Save best model =========
save_path = “best_mlp_optuna.pt”
torch.save({
    “state_dict”: {k: v.cpu() for k, v in best_model.state_dict().items()},
    “label_classes”: le.classes_,
    “params”: best_params,
    “in_dim”: in_dim,
    “num_classes”: num_classes,
}, save_path)
print(f”Saved best model to {save_path}“)'''

[I 2025-12-22 19:04:52,883] A new study created in memory with name: no-name-ef8babc1-f0f8-4fdb-b8dd-cc8d39033824
Best trial: 0. Best value: 0.145121:  10%|█         | 1/10 [00:10<01:34, 10.55s/it]

[I 2025-12-22 19:05:03,426] Trial 0 finished with value: 0.14512071337018695 and parameters: {'h1': 286, 'h2': 186, 'dropout': 0.40138168803582197, 'lr': 0.0015119336467641006, 'weight_decay': 1.8662266976517965e-05, 'use_class_weights': True}. Best is trial 0 with value: 0.14512071337018695.


Best trial: 1. Best value: 0.130999:  20%|██        | 2/10 [00:20<01:20, 10.07s/it]

[I 2025-12-22 19:05:13,163] Trial 1 finished with value: 0.13099892343793595 and parameters: {'h1': 458, 'h2': 248, 'dropout': 0.2917207594128889, 'lr': 0.014685885989200861, 'weight_decay': 3.860866271460548e-05, 'use_class_weights': False}. Best is trial 1 with value: 0.13099892343793595.


Best trial: 1. Best value: 0.130999:  30%|███       | 3/10 [00:30<01:09,  9.91s/it]

...

[I 2025-12-22 19:06:22,522] Trial 8 finished with value: 0.12060156149523599 and parameters: {'h1': 338, 'h2': 72, 'dropout': 0.3331553864281531, 'lr': 9.499535455183794e-05, 'weight_decay': 2.9985324349454373e-06, 'use_class_weights': False}. Best is trial 8 with value: 0.12060156149523599.


Best trial: 8. Best value: 0.120602: 100%|██████████| 10/10 [01:39<00:00,  9.93s/it]


[I 2025-12-22 19:06:32,214] Trial 9 pruned. 
Best trial: 8
Best value (val_loss): 0.12060156149523599
Best params: {'h1': 338, 'h2': 72, 'dropout': 0.3331553864281531, 'lr': 9.499535455183794e-05, 'weight_decay': 2.9985324349454373e-06, 'use_class_weights': False}
Val_Acc	Val_F1	Test_Acc	Test_F1
0.9571	0.9419	0.8207	0.7290
                  precision    recall  f1-score   support

     AVN/AV Ring     0.7725    0.8030    0.7874       203
         IFT/SAN     0.9448    0.7808    0.8550       219
        Inner-LV     0.9268    0.7166    0.8083      1768
        Inner-RV     0.7834    0.7445    0.7634      1127
      Left Atria     0.7045    0.8953    0.7885       812
Mus. Valve Leaf.     0.7727    0.5714    0.6570       476
             OFT     0.0000    0.0000    0.0000       474
        Outer-LV     0.9438    0.9560    0.9499      3814
        Outer-RV     0.9292    0.8514    0.8886      1110
     Right Atria     0.9826    0.8246    0.8967      1710
   Subepicardial     0.6147    0.696

/users/ajain59/.conda/envs/scGPT/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/users/ajain59/.conda/envs/scGPT/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/users/ajain59/.conda/envs/scGPT/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital

'# ========= 7) Save best model =========\nsave_path = “best_mlp_optuna.pt”\ntorch.save({\n    “state_dict”: {k: v.cpu() for k, v in best_model.state_dict().items()},\n    “label_classes”: le.classes_,\n    “params”: best_params,\n    “in_dim”: in_dim,\n    “num_classes”: num_classes,\n}, save_path)\nprint(f”Saved best model to {save_path}“)'